# Dribbling winger rating model
**Bundesliga 2023-24 · data: Impect open data**

<img src="https://raw.githubusercontent.com/ImpectAPI/open-data/main/img/impect_logo_black.svg" width="160" align="left">
<br clear="all">

Data © Impect, free for non-commercial use: https://github.com/ImpectAPI/open-data. The first code cell downloads it from Impect's GitHub for you.

**What this does:** gets the data. The first time you run it, it downloads Impect's free data straight from their GitHub (about 84MB). After that it skips the download. `prep.py` then adds up every player's 306 matches and turns them into per-90 numbers. `helpers.py` only handles how tables and charts look.

In [ ]:
from get_data import download_data
from prep import season_per90, metric_definitions
from helpers import STAT_NAMES, show, plot_shortlist, compare, plot_radar

download_data()   # first run only: fetches the data from Impect's GitHub into ./data

players = season_per90().rename(columns=STAT_NAMES)   # one row per player, every stat per 90 minutes
metrics = metric_definitions()                         # Impect's 103 stats and what they mean

## 1. The profile
A winger who beats his man and carries the ball into the box. The attributes, in order of importance:

1. Beats his man 1v1
2. Carries the ball forward
3. Takes defenders out of the game
4. Creates for others
5. Gets into scoring positions

## 2. The data

In [ ]:
print("Matches: 306 (the full 2023-24 Bundesliga season)")
print(f"Players: {len(players)}")
print(f"Stats:   {len(metrics)}")

## 3. The stats available
**What this does:** lists every stat Impect gives us, grouped, with one line on what each one measures. Read the definitions before you pick anything: the same word can mean different things to different data providers.

In [ ]:
show(metrics[["group", "metric", "short"]].rename(columns={"metric": "stat", "short": "what it measures"}))

In [ ]:
def find(word):
    # search Impect's stat names for a word, and show the full definition
    matches = metrics[metrics["metric"].str.contains(word, case=False)]
    return show(matches[["metric", "definition"]])


find("dribbl")   # what does Impect actually mean by a dribble?

> **What counts as a dribble at Impect?** From their documentation: *"Our data operators have to tag the event if one of the following rules are met: when the player carries the ball more than 2 steps, when the pressure changes significantly during the player's possession, or when the number of opponents changes during the player's possession."*
>
> So a "dribble" here is closer to a **carry** than to a take-on. To measure beating a man 1v1, we need ground duels won as well.

## 4. From profile to data
**What this does:** the profile written as data, the way Sahil's Model 1 does it. Each stat gets a **weight out of 100**: how much it counts towards the final rating. The comments show which part of the profile each stat measures. The weights must add up to 100.

In [ ]:
WEIGHTS = {
    # beats his man 1v1
    "Ground duels won": 15,
    "Dribble threat": 15,

    # carries the ball forward
    "Distance carried towards goal": 25,

    # takes defenders out of the game
    "Defenders bypassed": 20,

    # creates for others
    "Shot-creating actions": 8,
    "Expected assists": 7,

    # gets into scoring positions
    "Packing xG": 10,
}

print(f"Total: {sum(WEIGHTS.values())}%")
assert sum(WEIGHTS.values()) == 100, "The weights must add up to 100%" 

## 5. The clean-up
**What this does:** picks who we compare against (wingers and attacking midfielders) and removes anyone under 900 minutes, so one good afternoon can't top the list. Watch the minutes column change.

In [ ]:
POSITIONS = ["LEFT_WINGER", "RIGHT_WINGER", "ATTACKING_MIDFIELD"]   # who we compare against
MIN_MINUTES = 900                                                      # about 10 full games

wide_players = players[players["position"].isin(POSITIONS)]
columns = ["player", "team", "minutes", "Dribble threat"]

print("Top 5 for dribble threat, no minutes filter:")
display(show(wide_players.sort_values("Dribble threat", ascending=False)[columns].head(5), decimals=3))

wingers = wide_players[wide_players["minutes"] >= MIN_MINUTES]
print(f"With {MIN_MINUTES}+ minutes ({len(wingers)} players):")
display(show(wingers.sort_values("Dribble threat", ascending=False)[columns].head(5), decimals=3))

## 6. The rating
**First, what's a z-score?** It's how far a player is from the average, measured in "typical gaps". 0 is average, +1 is well above, -1 is well below. It lets us compare stats with different units: duels, metres, xG.

In [ ]:
stat = "Ground duels won"
average = wingers[stat].mean()   # the average winger
spread = wingers[stat].std()     # the typical gap from that average

example = wingers[wingers["player"].isin(["Leroy Sané", "Brajan Gruda", "Jadon Sancho"])][["player", stat]].copy()
example["vs average"] = example[stat] - average
example["z-score"] = example["vs average"] / spread

print(f"Average: {average:.2f} per 90   |   typical gap: {spread:.2f}")
show(example)

**What the model does, step by step** (Sahil's Model 1):
1. Turn every stat into a z-score.
2. Cap it at ±3, so one freak number can't decide the ranking.
3. Multiply each stat by its weight (%) and add them up.
4. Turn the total into a **0-100 rating**: "better than X% of the pool".

In [ ]:
def rate_players(wingers, weights):
    shortlist = wingers[["player", "team", "age", "minutes"]].copy()
    shortlist["age"] = shortlist["age"].astype(int)
    total = 0

    for stat, weight in weights.items():
        z = (wingers[stat] - wingers[stat].mean()) / wingers[stat].std()   # 1. z-score
        z = z.clip(-3, 3)                                                   # 2. cap at +/-3
        shortlist[stat] = z
        total = total + weight / 100 * z                                    # 3. weight and add

    shortlist["rating"] = (total.rank(pct=True) * 100).round().astype(int)  # 4. 0-100
    shortlist["score"] = total
    return shortlist.sort_values("score", ascending=False).reset_index(drop=True)


shortlist = rate_players(wingers, WEIGHTS)

top10 = shortlist.head(10)[["player", "team", "age", "minutes", "rating"]]
top10.insert(0, "rank", range(1, 11))
show(top10)

In [ ]:
plot_shortlist(shortlist, highlight="Brajan Gruda",
               title="Dribbling winger shortlist · Bundesliga 2023-24 · data: Impect")

## 7. Change the weights
**What this does:** the manager wants goals from wide. Packing xG goes from 10% to 30%, so other weights come down to keep the total at 100%. Same data, different shortlist.

In [ ]:
goals_from_wide = {
    "Ground duels won": 10,                # was 15
    "Dribble threat": 10,                  # was 15
    "Distance carried towards goal": 20,   # was 25
    "Defenders bypassed": 15,              # was 20
    "Shot-creating actions": 8,            # same
    "Expected assists": 7,                 # same
    "Packing xG": 30,                      # was 10
}
assert sum(goals_from_wide.values()) == 100

new_shortlist = rate_players(wingers, goals_from_wide)
show(compare(shortlist, new_shortlist, track=["Brajan Gruda"]), highlight="Brajan Gruda")

## 8. Why him?
**What this does:** if the sporting director asks "why him?", you need one sentence. This shows one player's percentile in each of the five key areas: 90 means better than 90% of the wingers in the pool, and 50 is the middle. Where an area has two stats, it shows the average of the two.

In [ ]:
AREAS = {
    "Beats his man":       ["Ground duels won", "Dribble threat"],
    "Carries forward":     ["Distance carried towards goal"],
    "Takes defenders out": ["Defenders bypassed"],
    "Creates":             ["Shot-creating actions", "Expected assists"],
    "Scoring positions":   ["Packing xG"],
}

player = "Brajan Gruda"

# percentile on every stat: 90 = better than 90% of the wingers in the pool
percentiles = shortlist.set_index("player")[list(WEIGHTS)].rank(pct=True) * 100

# one number per area: the average of that area's stats
areas = {area: round(percentiles.loc[player, stats].mean()) for area, stats in AREAS.items()}

plot_radar(areas, f"{player} · percentile in the pool · data: Impect")

In [ ]:
# the one-sentence answer: where he's strong, and what to watch him for
strengths = [f"{area} ({score})" for area, score in areas.items() if score >= 75]
watch_for = [f"{area} ({score})" for area, score in areas.items() if score < 50]

print("Strengths:", ", ".join(strengths))
print("Watch for:", ", ".join(watch_for))

## 9. What the model can't tell you
- **Team strength:** four of the top five play for Bayern or Leverkusen. Next step: adjust for team quality.
- **League strength:** fine inside one league. Comparing across leagues needs league weightings.
- **Position isn't role:** the pool mixes wingers with attacking midfielders.
- **Sample size:** some players only just clear 900 minutes.
- **Budget fit:** age, contract and market value come from elsewhere, for example Transfermarkt.
- **What event data can't see:** pace, decisions, mentality. That's what video and live scouting are for.

*A model doesn't find players. It tells you who to go and watch.*